# Phase 1 EDA ? t08-v1.0-20261010
Offline, no training. Read the versioned release; keep master, ranking and ML denominators separate. See docs/phase-1-completion.md for Vietnamese instructions.

In [1]:
from pathlib import Path
import json
import pandas as pd
root = Path.cwd()
if root.name == "notebooks": root = root.parent
p = root / "data/processed/phase1_v1_20261010"
meta = json.loads((p / "metadata.json").read_text(encoding="utf-8"))
master = pd.read_parquet(p / "cves.parquet")
ml = pd.read_parquet(p / "ml_cohort.parquet")
ranking = pd.read_parquet(p / "ranking_cohort.parquet")
print(meta["dataset_version"])
print(json.dumps(meta["statistics"], indent=2))

t08-v1.0-20261010
{
  "master": 71653,
  "unique_cve_ids": 71653,
  "rejected": 2884,
  "missing_epss": 2884,
  "scope_status": {
    "needs_review": 68362,
    "excluded": 2895,
    "included": 396
  },
  "web": 388,
  "mobile": 8,
  "ranking": 396,
  "ranking_kev": 3,
  "ml_before_duplicates": 67912,
  "ml": 67242,
  "excluded_duplicate_rows": 670,
  "splits": {
    "train": 28815,
    "validation": 19606,
    "test": 18821
  },
  "scoped_ml_splits": {
    "train": 219,
    "test": 108,
    "validation": 69
  },
  "base_scores_checked": 67912,
  "base_score_mismatches": 0,
  "shared_product_groups": 5944,
  "case_pairs": 5
}


## Source labels and missing data
Observed Base scores are labels, not exploitation probability. KEV absence is not proof of no exploitation.

In [2]:
print(master[["description", "cvss_vector", "epss", "is_kev"]].isna().sum())
print(master.groupby(["cvss_source", "cvss_type"], dropna=False).size().sort_values(ascending=False).head(12))
print("Modified after published:", int((master.last_modified > master.published).sum()))
print("Last modified years:", master.last_modified.dt.year.value_counts().sort_index().to_dict())

description       0
cvss_vector    3741
epss           2884
is_kev            0
dtype: int64
cvss_source                           cvss_type
nvd@nist.gov                          Primary      49670
134c704f-9b21-4f2e-91b3-4a467353bcc0  Secondary     5732
<NA>                                  <NA>          3741
audit@patchstack.com                  Secondary     2613
security@wordfence.com                Secondary     2111
secure@microsoft.com                  Secondary     1734
psirt@adobe.com                       Secondary     1077
secalert_us@oracle.com                Secondary      614
security-advisories@github.com        Secondary      561
nvd@nist.gov                          Secondary      393
twcert@cert.org.tw                    Secondary      226
secure@intel.com                      Secondary      186
dtype: int64
Modified after published: 71312
Last modified years: {2023: 2037, 2024: 735, 2025: 93, 2026: 68788}


## Scope and support
All ML metrics and scoped Web/mobile metrics must be reported separately. Zero-support classes remain in the fixed mapping.

In [3]:
print(master.scope_status.value_counts())
print(ml[["split", "domain_tags"]].explode("domain_tags").groupby(["split", "domain_tags"]).size())
support = pd.read_csv(p / "label_support.csv")
print(support.to_string(index=False))

scope_status
needs_review    68362
excluded         2895
included          396
Name: count, dtype: int64
split       domain_tags
test        web            108
train       mobile           8
            web            211
validation  web             69
dtype: int64
       cohort      split metric label  support  size
       all_ml      train     AV     N    21730 28815
       all_ml      train     AV     A      491 28815
       all_ml      train     AV     L     6378 28815
       all_ml      train     AV     P      216 28815
       all_ml      train     AC     L    27790 28815
       all_ml      train     AC     H     1025 28815
       all_ml      train     PR     N    16123 28815
       all_ml      train     PR     L     9746 28815
       all_ml      train     PR     H     2946 28815
       all_ml      train     UI     N    18691 28815
       all_ml      train     UI     R    10124 28815
       all_ml      train      S     U    22904 28815
       all_ml      train      S     C     591

## Duplicate and product-overlap limitations
Approximate retrieval may miss similar text. Shared products remain allowed in the temporal protocol; no unseen-product claim.

In [4]:
print(json.dumps(meta["near_duplicate_audit"], indent=2))
print(pd.read_csv(p / "product_overlap.csv").sort_values("rows", ascending=False).head(15).to_string(index=False))
print("Selected KEV in ranking:", int(ranking.is_kev.sum()))
print("EPSS:", meta["epss_date"], "KEV:", meta["kev_date_released"])

{
  "algorithm": "32 MinHash / 8x4 bands / 3-token shingles / exact Jaccard verification",
  "threshold": 0.95,
  "seed": 42,
  "candidate_pairs": 400471,
  "verified_edges": 71,
  "skipped_large_buckets": 16,
  "max_bucket": 200,
  "unique_normalized_descriptions": 63948,
  "limitation": "Approximate retrieval; near/semantic duplicates may remain. Shared products allowed in temporal evaluation."
}
                 product_groups  rows                splits
           o:linux:linux_kernel  4669 test|train|validation
               o:google:android  2007 test|train|validation
          o:debian:debian_linux  1021 test|train|validation
                  o:apple:macos   935 test|train|validation
         o:fedoraproject:fedora   916 test|train|validation
o:microsoft:windows_server_2022   819 test|train|validation
o:microsoft:windows_server_2019   803 test|train|validation
    o:microsoft:windows_11_22h2   738 test|train|validation
    o:microsoft:windows_10_22h2   712 test|train|validatio